# Qwen3-8B Bloom classifier with Unsloth

This notebook fine-tunes `unsloth/Qwen3-8B-bnb-4bit` with LoRA for the `QuestionSemantics` JSON contract used by the prediction engine. It validates the dataset before loading the GPU model, saves a compact LoRA adapter, optionally exports a `q4_k_m` GGUF for Ollama, and runs one inference check.

> The bundled examples are synthetic pipeline tests, not production training evidence. Use lecturer-adjudicated records for a real experiment.

Before continuing, select **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Verify the Colab GPU runtime.
import subprocess

subprocess.run(["nvidia-smi"], check=True)
print("A GPU is attached; CUDA is checked again after dependency installation.")

## Get the project

Push this training folder to GitHub before running the clone cell. Change `BRANCH` after merging it into another branch. If the repository is private, use Colab's GitHub integration or clone it into `/content/R26-SE-025` yourself.

In [ ]:
# 2. Clone the repository once.
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/akilaManu-MaHiTo/R26-SE-025.git"
BRANCH = "feat/bloom-model"
REPO_DIR = Path("/content/R26-SE-025")
TRAINING_DIR = REPO_DIR / "V2_QuestionExamPredictionEngine" / "unsloth_training"

if not TRAINING_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
print("Training directory:", TRAINING_DIR)
assert (TRAINING_DIR / "train_bloom.py").is_file(), "Training scaffold was not found."

In [ ]:
# 3. Install current Unsloth/TRL dataset dependencies. This can take a few minutes.
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(TRAINING_DIR / "requirements-colab.txt")],
    check=True,
)
import torch
assert torch.cuda.is_available(), "The installed PyTorch build cannot access CUDA."
print("Training dependencies installed. GPU:", torch.cuda.get_device_name(0))

## Select and validate data

Leave `USE_EXAMPLE_DATA=True` only for a one-epoch smoke test. For real training, upload adjudicated `train.jsonl` and `validation.jsonl`, set their paths below, and change it to `False`.

In [ ]:
# 4. Choose data and training settings.
USE_EXAMPLE_DATA = True
TRAIN_FILE = TRAINING_DIR / "data" / ("example_train.jsonl" if USE_EXAMPLE_DATA else "train.jsonl")
VALIDATION_FILE = TRAINING_DIR / "data" / ("example_validation.jsonl" if USE_EXAMPLE_DATA else "validation.jsonl")

MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"
EPOCHS = 1.0 if USE_EXAMPLE_DATA else 3.0
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
EXPORT_GGUF = True

print("Train:", TRAIN_FILE)
print("Validation:", VALIDATION_FILE)
print("Example-only smoke test:", USE_EXAMPLE_DATA)

In [ ]:
# 5. Validate JSON structure, controlled labels, duplicates, and split leakage.
validate_command = [
    sys.executable,
    str(TRAINING_DIR / "validate_dataset.py"),
    str(TRAIN_FILE),
    str(VALIDATION_FILE),
]
if not USE_EXAMPLE_DATA:
    validate_command.insert(2, "--require-adjudicated")
subprocess.run(validate_command, check=True)

## Persist outputs in Google Drive

The GGUF can be several gigabytes and Colab storage is temporary. The next cell mounts Drive and writes outputs there. Set `SAVE_TO_DRIVE=False` if you only want temporary `/content` output.

In [ ]:
# 6. Choose persistent output paths.
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/qwen3_bloom_outputs")
else:
    OUTPUT_ROOT = Path("/content/qwen3_bloom_outputs")

LORA_DIR = OUTPUT_ROOT / "qwen3-bloom-lora"
GGUF_DIR = OUTPUT_ROOT / "qwen3-bloom-gguf"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output root:", OUTPUT_ROOT)

## Train

This cell invokes the repository's training script. The example run proves that the pipeline works, but its resulting adapter must not be used as an accuracy result or production model.

In [ ]:
# 7. Fine-tune and optionally export GGUF.
train_command = [
    sys.executable,
    str(TRAINING_DIR / "train_bloom.py"),
    "--train", str(TRAIN_FILE),
    "--validation", str(VALIDATION_FILE),
    "--model", MODEL_NAME,
    "--output-dir", str(LORA_DIR),
    "--gguf-dir", str(GGUF_DIR),
    "--epochs", str(EPOCHS),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation", str(GRADIENT_ACCUMULATION),
    "--learning-rate", str(LEARNING_RATE),
]
if USE_EXAMPLE_DATA:
    train_command.append("--allow-example-data")
if EXPORT_GGUF:
    train_command.append("--export-gguf")

print("Starting training...")
subprocess.run(train_command, check=True)
print("Training complete. LoRA:", LORA_DIR)
if EXPORT_GGUF:
    print("GGUF:", GGUF_DIR)

## Inference smoke test

Reload the saved adapter and classify an unseen question. A valid JSON shape only proves integration; evaluate accuracy separately on an untouched lecturer-labelled test set.

In [ ]:
# 8. Reload the adapter and test one question.
from unsloth import FastLanguageModel

inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(LORA_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(inference_model)

messages = [
    {
        "role": "system",
        "content": "Classify the required cognitive process using Revised Bloom's Taxonomy. Judge the work needed for a complete answer, not the command verb alone. Return only JSON with level, topic, subtopic, confidence, and reason.",
    },
    {
        "role": "user",
        "content": "COURSE: IT2040 - Database Management Systems\nQUESTION: Compare two normalization designs and justify which better preserves dependencies.\nRUBRIC_CRITERIA: Evaluates both designs against lossless join and dependency preservation.",
    },
]
inputs = inference_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")
outputs = inference_model.generate(
    input_ids=inputs,
    max_new_tokens=192,
    do_sample=False,
)
generated = outputs[0, inputs.shape[-1]:]
print(inference_tokenizer.decode(generated, skip_special_tokens=True))

In [ ]:
# 9. Show exported artifacts.
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(OUTPUT_ROOT)}  {path.stat().st_size / (1024 ** 2):.1f} MB")

## Use the GGUF with Ollama

Download or copy the exported `.gguf` to your machine. Put it beside `Modelfile.example`, replace the `FROM` filename, then run:

```bash
ollama create qwen3-bloom:latest -f Modelfile.example
ollama run qwen3-bloom:latest
```

Keep this specialized tag for Bloom classification only. The application still needs separate `BLOOM_MODEL` and general-model routing before this should replace classification in production.